# **CMSE 381 Final Project**

## Group Members: Alex Chen and Kevin Chiang

## Section 01

## Date: April 24, 2026

## **Project Title:** 

## Predicting Student Academic Performance Using Behavioral and Demographic Data

## **Background and Motivation**

Student academic performance is influenced by a combination of behavioral, environmental, and socioeconomic factors. Understanding these relationships is important for improving educational outcomes and identifying at-risk students early.

## **Research Questions**

* Which factors most strongly influence exam scores?
* Can we accurately predict exam scores using regression models?
* Can we classify students into performance categories, like high vs low?
* Do behavioral factors, including study hours and sleep, matter more than demographic factors?

## **Methodology**

We approached this problem in three steps:

* Data preprocessing
* Regression modeling (predict Exam_Score)
* Classification modeling (high vs low performance)
* Additional method: Feature importance + cross-validation

## **Data**

The dataset contains student-level data with variables such as:

* Hours_Studied: weekly study hours
* Attendance: % attendance
* Sleep_Hours
* Previous_Scores
* Motivation_Level
* Family_Income
* Exam_Score (target)

### *Import Libraries*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('default')
sns.set_theme()

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report

### *Load Data*

In [ ]:
df = pd.read_csv("StudentPerformanceFactors.csv")
df.head()

### *Preprocessing*

Convert categorical variables using one-hot encoding.

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop("Exam_Score", axis=1)
y = df_encoded["Exam_Score"]

## **Models for Regression**

### Goal:

* Predict Exam_Score


### Models Used:

* Linear Regression
* Random Forest Regressor

### Why:

* Linear: interpretable baseline
* Random Forest: captures nonlinear relationships

### *Train/Test Split*

In [ ]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y, test_size=0.2, random_state=42)

### *Linear Regression*

In [ ]:
lr = LinearRegression()
lr.fit(X_train_reg, y_train_reg)
y_pred_lr = lr.predict(X_test_reg)

rmse_lr = np.sqrt(mean_squared_error(y_test_reg, y_pred_lr))
print("Linear Regression RMSE:", rmse_lr)

### *Random Forest Regression*

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_reg, y_train_reg)
y_pred_rf_reg = rf.predict(X_test_reg)
rmse_rf = np.sqrt(mean_squared_error(y_test_reg, y_pred_rf_reg))
print("Random Forest RMSE:", rmse_rf)

## **Models for Classification**

### Goal:

Classify students into:

* High performance (Exam_Score $\geq 70$)
* Low performance ($< 70$)

### *Create Target*

In [ ]:
df_encoded["High_Performance"] = (df_encoded["Exam_Score"] >= 70).astype(int)

X_class = df_encoded.drop(["Exam_Score", "High_Performance"], axis=1)
y_class = df_encoded["High_Performance"]

### *Logistic Regression*

In [ ]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_class, y_class, test_size=0.2, random_state=42)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_clf = scaler.fit_transform(X_train_clf)
X_test_clf = scaler.transform(X_test_clf)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_clf, y_train_clf)

y_pred = log_model.predict(X_test_clf)

print("Accuracy:", accuracy_score(y_test_clf, y_pred))
print(classification_report(y_test_clf, y_pred))

### *Random Forest Classifier*

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train_clf, y_train_clf)

y_pred_rf = rf_clf.predict(X_test_clf)

print("RF Accuracy:", accuracy_score(y_test_clf, y_pred_rf))

## **Other Methods Used**

### *K-Fold Cross Validation*

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(rf, X, y, cv=kf, scoring='neg_mean_squared_error')
rmse_scores = np.sqrt(-scores)

print("CV RMSE:", rmse_scores.mean())

### *Feature Importance*

In [ ]:
importances = rf.feature_importances_
features = X.columns

feat_df = pd.DataFrame({"Feature": features, "Importance": importances})
feat_df = feat_df.sort_values(by="Importance", ascending=False)

print(feat_df.head())

## **Results**

### Regression Results

#### What are we trying to do?

We aim to predict students’ Exam_Score using regression models and evaluate how accurately different models perform.

#### Model Performance

* Linear Regression RMSE: $1.80$
* Random Forest RMSE: $2.23$
* Cross-Validation RMSE (RF): $2.40$

## **Visualizations**

In [ ]:
plt.figure()
plt.scatter(y_test_reg, y_pred_lr)
plt.plot([y_test_reg.min(), y_test_reg.max()],
         [y_test_reg.min(), y_test_reg.max()],
         linestyle='--')
plt.xlabel("Actual Exam Score")
plt.ylabel("Predicted Exam Score")
plt.title("Linear Regression: Actual vs Predicted")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
plt.scatter(y_test_reg, y_pred_rf_reg)
plt.xlabel("Actual Exam Score")
plt.ylabel("Predicted Exam Score")
plt.title("Random Forest: Actual vs Predicted")
plt.show()

### **Interpretation**

* Surprisingly, Linear Regression outperforms Random Forest, with a lower RMSE ($1.80$ vs $2.23$).
* This suggests that the relationship between features and exam score is largely linear, rather than highly complex or nonlinear.
* The cross-validation RMSE (~$2.40$) confirms that the Random Forest model is slightly less stable and generalizes worse than Linear Regression.

Linear Regression achieved a lower RMSE than Random Forest, indicating better predictive performance.

These results suggest that the relationship between features and exam score is largely linear, or that additional model complexity does not provide meaningful gains for this dataset. Linear Regression performs well because it models direct proportional relationships between variables such as study time and attendance.

This suggests either that relationships are mostly linear or that additional model complexity does not provide meaningful gains for this dataset.

## **Classification Results**

### What are we trying to do?

We classify students into:
* High Performance (Exam_Score $\geq 70$)
* Low Performance (Exam_Score $< 70$)

### Model Performance

Logistic Regression:
* Accuracy: $98.56$%
* Very high precision and recall for both classes

Random Forest Classifier:
* Accuracy: $91.53$%

### Interpretation

* Logistic Regression significantly outperforms Random Forest in classification, achieving an accuracy of approximately $98.6$% compared to $91.4$%.
* This suggests that the dataset is well-separated using linear boundaries, making a simple linear model highly effective.
* The very high accuracy may also indicate that the dataset is highly structured or contains strong predictive signals.
* The high recall values $(0.96-0.99)$ indicate that the model correctly identifies most students in both performance categories.
* This implies that student performance categories can be predicted with high confidence based on the available features.

Logistic Regression performs well because it assumes a linear relationship between the features and the probability of high performance, which appears to fit this dataset effectively.

In contrast, Random Forest, while still performing well, does not provide additional benefit, suggesting that complex nonlinear interactions are not as important for this classification task.

Overall, this indicates that the distinction between high- and low-performing students can largely be explained by simple combinations of key variables such as attendance, study time, and previous scores.

## **Other Results (Feature Importance)**

| Feature           | Importance |
| ----------------- | ---------- |
| Attendance        | 0.382      |
| Hours_Studied     | 0.246      |
| Previous_Scores   | 0.092      |
| Tutoring_Sessions | 0.036      |
| Physical_Activity | 0.027      |

Attendance, Hours Studied, and Previous Scores are the most influential variables.

This makes intuitive sense, as students who have performed well in the past and dedicate more time to studying are more likely to achieve higher exam scores. Attendance is also important because consistent class participation likely improves understanding of the material.

Behavioral factors appear to have stronger predictive importance than demographic variables in this dataset.

### *Visualization*

In [ ]:
top_features = feat_df.nlargest(10, "Importance")
plt.figure()
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Feature Importance")

plt.title("Top Predictors of Student Performance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
residuals = y_test_reg - y_pred_lr
plt.scatter(y_pred_lr, residuals, alpha=0.6)
plt.axhline(y=0, linestyle='--')
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Plot (Linear Regression)")
plt.tight_layout()
plt.show()

### **Interpretation**

* Attendance is the most important feature, accounting for approximately $38$% of the model’s total feature importance score.
* Hours studied is the second most important factor.
* Academic history (Previous Scores) also plays a meaningful role.
* Lifestyle factors (physical activity, tutoring) have smaller but noticeable effects.

## **Discussion and Conclusion**

### Discussion on Regression Results

* Linear Regression performed better than Random Forest, indicating that:
  * Relationships in the dataset are mostly linear
  * There is limited benefit from complex nonlinear models
* The low RMSE (~$1.8$) suggests that the model achieves relatively accurate predictions with small average errors.

### Discussion on Classification Results

* Logistic Regression achieved exceptionally high accuracy (~$98.6$%)
* This implies:
  * Clear separation between high and low performers
  * Strong predictive signals in the dataset
* Random Forest performed worse, suggesting that additional model complexity does not provide significant benefits for this dataset.
 
### Discussion on Other Results

* Attendance and study habits dominate performance prediction.
* Socioeconomic and demographic factors appear less influential compared to behavioral ones.
* This suggests that student behavior is more important than background in this dataset.

## **Conclusion and Future Steps**

### Final Conclusions (Quantitative Answers)

* Attendance is the most influential predictor, contributing approximately ~$38$% of total feature importance in the Random Forest model.
* Hours studied accounts for roughly ~$24$% of the model’s predictive power, making it the second most important factor.
* Linear Regression achieved a lower prediction error, RMSE $\approx 1.80$, compared to Random Forest, RMSE $\approx 2.24$, indicating that a linear model fits this dataset better.
* Logistic Regression achieved very high classification accuracy ~$98.6$%, while Random Forest achieved ~$91.4$%, both indicating strong performance.

In the linear regression model, Hours Studied has a positive coefficient of approximately 0.29, indicating that each additional hour studied is associated with about a 0.3-point increase in exam score, holding other variables constant.
The better performance of Linear Regression suggests that the relationship between the predictors and exam scores is largely linear, meaning that increases in variables such as study time and attendance lead to relatively proportional increases in performance.

Students with higher attendance and previous scores tend to have higher predicted exam scores in the model.

Overall, these results indicate that behavioral factors such as study time and attendance have a measurable and significant impact on academic performance.

## **Challenges**

* Dataset may be too clean or structured, leading to extremely high accuracy (possible overfitting risk)
* Many categorical variables required encoding, increasing dimensionality
* Lack of external validation dataset limits generalizability

## **Future Improvements**

* Apply more advanced models:
  * Gradient Boosting (XGBoost)
  * Neural Networks
* Perform feature engineering:
  * Interaction terms, like study hours and attendance
* Use regularization techniques (Ridge/Lasso)
* Collect more diverse or real-world data
* Perform deeper error analysis, where models fail

## **Author Contribution**

### **Alex Chen:**

* Data preprocessing
* Regression modeling
* Visualization

### **Kevin Chiang:**

* Classification modeling
* Cross-validation
* Discussion & slides

## **References**

Ayesha Siddiqa. Student Performance Dataset. Kaggle. https://www.kaggle.com/datasets/ayeshasiddiqa123/student-perfirmance